# 01 — Data Pipeline

Load, clean, and prepare raw data for analysis.

In [1]:
import sys
import os

# Adds to path
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Set plot style (try newer name first; fallback for older matplotlib)
# Reusable template
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully")

Libraries loaded successfully


In [7]:
from src.api_client import fetch_all_stocks

# Load from cache — no API calls if CSVs already in data/raw/
stocks = fetch_all_stocks()
print(stocks)
# Quick summary of what we have
for name, df in stocks.items():
    print(f"{name}: {len(df)} rows | {df.index.min().date()} to {df.index.max().date()}")

Saved IBM to cache (Alpha Vantage).
Saved Apple to cache (Alpha Vantage).
Saved Microsoft to cache (Alpha Vantage).
{'IBM':               open      high       low   close      volume
2025-10-23  264.95  285.5791  263.5623  285.00  16676394.0
2025-10-24  283.77  310.7500  282.2100  307.46  16914243.0
2025-10-27  307.80  313.5000  302.8800  313.09   9868151.0
2025-10-28  312.60  319.3500  311.4100  312.57   6044770.0
2025-10-29  312.79  314.3300  307.5200  308.21   4135948.0
...            ...       ...       ...     ...         ...
2026-03-12  247.10  250.0450  245.6400  247.68   5547724.0
2026-03-13  247.67  249.7200  244.7100  246.28   4338431.0
2026-03-16  247.87  252.2000  246.1000  249.25   5674228.0
2026-03-17  250.51  256.3900  250.0000  256.11   5840139.0
2026-03-18  254.16  258.2800  250.1600  251.60   5177047.0

[100 rows x 5 columns], 'Apple':                open     high       low   close      volume
2025-10-23  259.940  260.620  258.0101  259.58  32754941.0
2025-10-24  261.

In [ ]:
# Exploring data, looking to clean.
for name, df in stocks.items():
    print(f"\n{name}")
    print(f"  Missing values: {df.isnull().sum().sum()}")
    print(f"  Zero volume days: {(df['volume'] == 0).sum()}")
    print(f"  Duplicate dates: {df.index.duplicated().sum()}")
    print(df.describe().round(2))


MTN Group
  Missing values: 0
  Zero volume days: 0
  Duplicate dates: 0
           open      high       low     close       volume
count    251.00    251.00    251.00    251.00       251.00
mean   15202.35  15505.57  14981.70  15269.31   5647124.42
std     2527.11   2567.82   2469.14   2520.83   3395879.00
min    10375.00  10537.00   9952.00  10355.00    713054.00
25%    13164.50  13438.50  13005.00  13220.00   3511438.00
50%    15230.00  15580.00  15041.00  15357.00   5003176.00
75%    16990.00  17278.00  16774.00  17011.50   6565887.50
max    20801.00  21398.00  20403.00  20898.00  33684983.00

Naspers
  Missing values: 0
  Zero volume days: 0
  Duplicate dates: 0
            open       high        low      close       volume
count     251.00     251.00     251.00     251.00       251.00
mean   107869.33  109208.74  106657.74  107788.34   1995603.14
std     11245.50   11229.89   11295.89   11213.62   1263665.02
min     79953.20   86768.00   79643.00   84438.00    359481.00
25%    1

In [8]:
# Daily return = percentage change in closing price day to day.
# Using returns (not raw price) lets you compare stocks that trade at different price levels.
returns = pd.DataFrame()

for name, df in stocks.items():
    returns[name] = df['close'].pct_change() * 100  # as percentage

returns.dropna(inplace=True)
print(returns)

print("Daily returns (%) — last 5 days:")
print(returns.tail().round(3))
print("\nAverage daily return (%):")
print(returns.mean().round(3))
print("\nVolatility (std dev of daily returns, %):")
print(returns.std().round(3))

                 IBM     Apple  Microsoft  MTN Group   Naspers  Shoprite  \
2025-10-24  7.880702  1.248170   0.585907  -0.173663  1.143540 -0.981868   
2025-10-27  1.831133  2.279126   1.510666   0.179964  3.732309 -0.908683   
2025-10-28 -0.166086  0.070682   1.984874   2.994012 -1.703504  2.391911   
2025-10-29 -1.394888  0.260223  -0.095929  -1.697674  0.882773  0.820677   
2025-10-30  0.600240  0.630330  -2.915705   0.621008 -1.645531 -2.165029   
...              ...       ...        ...        ...       ...       ...   
2026-03-12 -0.478161 -1.936275  -0.745900  -1.414791  0.654654 -1.460160   
2026-03-13 -0.565245 -2.205192  -1.570199  -0.570776  3.423642  0.011607   
2026-03-16  1.205944  1.079482   1.112375   7.003444  0.496907  4.139265   
2026-03-17  2.752257  0.557709  -0.135017  -0.040875  0.504709 -0.772660   
2026-03-18 -1.760962 -1.687448  -1.907814   1.144960 -6.986547 -1.370171   

            FirstRand  Anglo American  
2025-10-24  -0.240558       -2.294659  
2025-10